# DEPRECATED — NON-CANONICAL
Do not execute this historical smoke wrapper. Start only with `00_campaign_control.ipynb` and follow `docs/canonical-colab-runbook.md`.

# EG-SEG-002 — real-data smoke gate
Run only in the same runtime after notebook 04 produced a valid compatibility receipt. Framework installation is never repeated here.

In [ ]:
import json
import re
import subprocess
from pathlib import Path

EDGEGUARD_EXPECTED_COMMIT = "REPLACE_WITH_REVIEWED_LOCAL_FIRST_COMMIT_SHA"
PROJECT_ROOT = Path("/content/edgeguard-road")
COMPAT = Path("/content/edgeguard-compatibility")
if not re.fullmatch(r"[0-9a-f]{40}", EDGEGUARD_EXPECTED_COMMIT):
    raise ValueError("Enter the exact reviewed local-first commit SHA")
if not PROJECT_ROOT.is_dir():
    raise RuntimeError("Run compatibility notebook 04 first in this runtime")
actual = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
dirty = subprocess.run(
    ["git", "-C", str(PROJECT_ROOT), "status", "--porcelain=v1"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
if actual != EDGEGUARD_EXPECTED_COMMIT or dirty:
    raise RuntimeError("Current runtime checkout identity is invalid")

In [ ]:
receipt_path = COMPAT / "compatibility_receipt.json"
if not receipt_path.is_file():
    raise RuntimeError("Current runtime has no successful compatibility receipt")
receipt = json.loads(receipt_path.read_text(encoding="utf-8"))
if receipt.get("project_commit") != EDGEGUARD_EXPECTED_COMMIT:
    raise RuntimeError("Compatibility receipt belongs to another commit")
if receipt.get("five_model_probe", {}).get("model_count") != 5:
    raise RuntimeError("Compatibility receipt lacks all five models")
if receipt["five_model_probe"].get("checkpoint_resume_verified") is not True:
    raise RuntimeError("Compatibility receipt lacks checkpoint round-trip")
INTERPRETER = Path(receipt["interpreter"])
if not INTERPRETER.is_file():
    raise RuntimeError("Compatibility interpreter is absent from this runtime")
MMSEG = Path("/content/edgeguard-mmseg") / (
    "mmseg-path-a" if receipt["selected_path"] == "hosted_current" else "mmseg-path-b"
)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/EdgeGuard")
subprocess.run(
    [
        str(INTERPRETER),
        str(PROJECT_ROOT / "scripts/data/inventory_edgeguard_storage.py"),
        "--external-root",
        str(DRIVE_ROOT),
        "--dry-run",
        "--require-cityscapes-reusable",
    ],
    check=True,
)

In [ ]:
MANIFESTS = DRIVE_ROOT / "manifests/cityscapes/fine/v1"
POLICY = MANIFESTS / "split-policy-v1"
if not POLICY.exists():
    subprocess.run(
        [
            str(INTERPRETER),
            str(PROJECT_ROOT / "scripts/rebuild_cityscapes_splits.py"),
            "--dataset-manifest",
            str(MANIFESTS / "dataset_manifest.json"),
            "--group-summary",
            str(MANIFESTS / "group_summary.json"),
            "--output-directory",
            str(POLICY),
        ],
        check=True,
    )
stage_command = [
    str(INTERPRETER),
    str(PROJECT_ROOT / "scripts/stage_cityscapes_training.py"),
    "--dataset-root",
    str(DRIVE_ROOT / "datasets/cityscapes/fine/v1"),
    "--dataset-manifest",
    str(MANIFESTS / "dataset_manifest.json"),
    "--split-policy-manifest",
    str(POLICY / "policy_selected_split.json"),
    "--drive-bundle-directory",
    str(DRIVE_ROOT / "datasets/cityscapes/fine/bundles"),
    "--cache-directory",
    "/content/edgeguard-data-cache",
    "--staged-dataset-root",
    "/content/edgeguard-cityscapes-fine",
]
subprocess.run([*stage_command, "--dry-run"], check=True)
subprocess.run(stage_command, check=True)

In [ ]:
subprocess.run(
    [
        str(INTERPRETER),
        str(PROJECT_ROOT / "scripts/train/run_semantic_smoke.py"),
        "--interpreter",
        str(INTERPRETER),
        "--project-root",
        str(PROJECT_ROOT),
        "--project-commit",
        EDGEGUARD_EXPECTED_COMMIT,
        "--config-root",
        str(PROJECT_ROOT / "configs/training/segmentation"),
        "--mmseg-checkout",
        str(MMSEG),
        "--dataset-root",
        "/content/edgeguard-cityscapes-fine",
        "--dataset-manifest",
        str(MANIFESTS / "dataset_manifest.json"),
        "--split-policy-manifest",
        str(POLICY / "policy_selected_split.json"),
        "--run-root",
        "/content/edgeguard-runs/EG-SEG-002",
        "--drive-root",
        str(DRIVE_ROOT),
    ],
    check=True,
)
summary = json.loads(Path("/content/edgeguard-runs/EG-SEG-002/smoke_summary.json").read_text())
if summary["status"] != "ready_for_common_screening":
    raise RuntimeError("EG-SEG-002 smoke promotion gate did not pass")
summary

## Stop
Do not start common screening here. SMIYC and full Fishyscapes Lost & Found remain inaccessible.